# 05c — Patch: Re-solve Scenario B, p=8000 to Optimal

Only Round 2 config that returned `Not Solved` (hit the 180s cap without proving the 2% gap).
Everything else in Round 2 (11/12 configs) is already `Optimal` -- this notebook re-solves
just this one config with a longer time budget (600s) and overwrites its row in the saved
results, so the final results table is 12/12 Optimal instead of 11/12 + a caveat.

## 0. Setup and reload

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from scipy.spatial import cKDTree
import pulp
import os
import time

os.environ.setdefault("SHAPE_RESTORE_SHX", "YES")

BASE_CANDIDATES = [
    "/Users/alexia/Documents/CASA/Dissertation",
    os.path.abspath(os.path.join(os.getcwd(), "..")),
]
BASE = next(
    (b for b in BASE_CANDIDATES
     if os.path.exists(os.path.join(b, "05_processed/p_median_results.csv"))),
    BASE_CANDIDATES[0],
)
print("Using BASE:", BASE)

demand_london = pd.read_csv(os.path.join(BASE, "05_processed/demand_london.csv"))
seff_london   = pd.read_csv(os.path.join(BASE, "05_processed/seff_london.csv"))
imd_london    = pd.read_csv(os.path.join(BASE, "05_processed/imd_london_clean.csv"))
census_london = pd.read_csv(os.path.join(BASE, "05_processed/census_london_clean.csv"))
results_wide  = pd.read_csv(os.path.join(BASE, "05_processed/p_median_results.csv"))
run_status    = pd.read_csv(os.path.join(BASE, "05_processed/p_median_run_status.csv"))

results_wide["lsoa_code"] = results_wide["lsoa_code"].astype(str)
print("Current status for Scenario B, p=8000:")
print(run_status[(run_status["scenario"] == "B") & (run_status["p"] == 8000)])


Using BASE: /Users/alexia/Documents/CASA/Dissertation
Current status for Scenario B, p=8000:
  scenario     p          K      status   slack_total  slack_frac  \
5        B  8000  16.074656  Not Solved  34841.871134    0.100273   

      objective  M1_avg_dist_m  M2_coverage_800m  M3_imd_gap   M4_gini  
5  2.269086e+07      66.457771          0.984476    0.060143  0.129766  


In [2]:
lsoa_boundaries = gpd.read_file(os.path.join(BASE, "03_data/demand/spatial/LSOA_2021_EW_BGC_V5.shp"))
if lsoa_boundaries.crs is None:
    lsoa_boundaries = lsoa_boundaries.set_crs(epsg=27700)
elif lsoa_boundaries.crs.to_epsg() != 27700:
    lsoa_boundaries = lsoa_boundaries.to_crs(epsg=27700)

london_codes = set(demand_london["lsoa_code"])
lsoa_london = lsoa_boundaries[lsoa_boundaries["LSOA21CD"].isin(london_codes)].copy()
lsoa_london = lsoa_london.rename(columns={"LSOA21CD": "lsoa_code"})[["lsoa_code", "geometry"]]
lsoa_london["centroid"] = lsoa_london.geometry.centroid
lsoa_london["cx"] = lsoa_london["centroid"].x
lsoa_london["cy"] = lsoa_london["centroid"].y

lsoa_master = lsoa_london[["lsoa_code", "cx", "cy"]].merge(
    demand_london[["lsoa_code", "D_A", "D_B", "D_C", "D_D"]], on="lsoa_code", how="inner"
).merge(seff_london[["lsoa_code", "ej"]], on="lsoa_code", how="left").reset_index(drop=True)
lsoa_master["ej"] = lsoa_master["ej"].fillna(0)
lsoa_master = lsoa_master.merge(census_london[["lsoa_code", "Hi", "Ci"]], on="lsoa_code", how="left")
lsoa_master["Vi"] = lsoa_master["Hi"] * lsoa_master["Ci"]
lsoa_master = lsoa_master.merge(imd_london[["lsoa_code", "income_decile"]], on="lsoa_code", how="left")
lsoa_master["lsoa_code"] = lsoa_master["lsoa_code"].astype(str)

coords = lsoa_master[["cx", "cy"]].to_numpy()
p_primary = 250
K0 = lsoa_master["D_A"].sum() / (lsoa_master["ej"].sum() + p_primary)
print(f"K0 = {K0:.4f}")


K0 = 16.0747


## 1. Core functions (identical to Round 2 -- only time_limit changes)

In [3]:
def evaluate_allocation(demand_col, xj, K, lsoa_master, coords):
    Di = lsoa_master[demand_col].to_numpy()
    Vi = lsoa_master["Vi"].to_numpy()
    ej = lsoa_master["ej"].to_numpy()
    decile = lsoa_master["income_decile"].to_numpy()
    n = len(lsoa_master)

    capacity = ej + xj
    has_capacity = capacity > 0
    cap_positions = np.where(has_capacity)[0]
    tree_cap = cKDTree(coords[has_capacity])
    dist, nearest_pos = tree_cap.query(coords, k=1)
    assigned_j = cap_positions[nearest_pos]

    load_j = np.zeros(n)
    for i in range(n):
        load_j[assigned_j[i]] += Di[i]
    slack_j = K * capacity - load_j
    objective = float((Di * dist).sum())

    M1 = float((Vi * dist).sum() / Vi.sum())
    within_800 = (dist < 800).astype(float)
    M2 = float((Vi * within_800).sum() / Vi.sum())

    def coverage_for_decile(d):
        mask = decile == d
        if mask.sum() == 0 or Vi[mask].sum() == 0:
            return np.nan
        return float((Vi[mask] * within_800[mask]).sum() / Vi[mask].sum())

    cov_d1, cov_d10 = coverage_for_decile(1), coverage_for_decile(10)
    M3 = cov_d1 - cov_d10 if (cov_d1 is not None and cov_d10 is not None) else np.nan

    x = np.sort(1.0 / (dist + 1.0))
    nn = len(x)
    cum = np.cumsum(x)
    M4 = (nn + 1 - 2 * (cum.sum() / cum[-1])) / nn if cum[-1] > 0 else 0.0

    return {
        "sj": slack_j, "objective": objective, "n_lsoa_with_xj_gt_0": int((xj > 0).sum()),
        "M1_avg_dist_m": M1, "M2_coverage_800m": M2, "M3_imd_gap": M3, "M4_gini": M4,
    }


def solve_joint_p_median(demand_col, p, K, lsoa_master, coords,
                         Uj=150, k=40, feas_margin=0.02, allow_K_bump=True,
                         time_limit=600, frac_gap=0.02, msg=False):
    """Identical formulation to Round 2. time_limit raised 300s -> 600s and frac_gap
    tightened back to 2% (Round 2's original target) for this one patched config."""
    Di = lsoa_master[demand_col].to_numpy(dtype=float)
    ej = lsoa_master["ej"].to_numpy(dtype=float)
    n = len(lsoa_master)
    sum_Di, sum_ej = Di.sum(), ej.sum()

    K_used = K
    if allow_K_bump:
        K_min = sum_Di / (sum_ej + p)
        K_used = max(K, K_min * (1 + feas_margin))

    tree = cKDTree(coords)
    _, nbr = tree.query(coords, k=min(k, n))
    cand = [set(np.atleast_1d(row).tolist()) for row in nbr]
    for i in range(n):
        cand[i].add(i)

    prob = pulp.LpProblem("joint_p_median", pulp.LpMinimize)
    x = pulp.LpVariable.dicts("x", range(n), lowBound=0, upBound=Uj, cat="Integer")
    y = {(i, j): pulp.LpVariable(f"y_{i}_{j}", lowBound=0, upBound=1)
         for i in range(n) for j in cand[i]}
    s = pulp.LpVariable.dicts("s", range(n), lowBound=0)

    def d(i, j):
        return float(np.hypot(coords[i, 0] - coords[j, 0], coords[i, 1] - coords[j, 1]))

    max_dist = float(np.hypot(np.ptp(coords[:, 0]), np.ptp(coords[:, 1])))
    M = 100.0 * max_dist

    prob += (pulp.lpSum(Di[i] * d(i, j) * y[(i, j)] for (i, j) in y)
             + M * pulp.lpSum(s[j] for j in range(n)))

    for i in range(n):
        prob += pulp.lpSum(y[(i, j)] for j in cand[i]) == 1

    served_by = {j: [] for j in range(n)}
    for (i, j) in y:
        served_by[j].append(i)
    for j in range(n):
        if served_by[j]:
            prob += (pulp.lpSum(Di[i] * y[(i, j)] for i in served_by[j])
                     <= K_used * (ej[j] + x[j]) + s[j])

    prob += pulp.lpSum(x[j] for j in range(n)) == p

    status = prob.solve(pulp.PULP_CBC_CMD(msg=int(msg), timeLimit=time_limit, gapRel=frac_gap))
    xj = np.array([int(round(x[j].value() or 0)) for j in range(n)])
    sj = np.array([float(s[j].value() or 0) for j in range(n)])
    slack_total = float(sj.sum())
    return {
        "xj": xj, "sj": sj, "K_used": K_used, "status": pulp.LpStatus[status],
        "slack_total": slack_total, "slack_frac": slack_total / sum_Di if sum_Di else 0.0,
    }


## 2. Re-solve Scenario B, p=8000 with a 600s budget

In [4]:
t0 = time.time()
sol = solve_joint_p_median("D_B", p=8000, K=K0, lsoa_master=lsoa_master, coords=coords, time_limit=600, frac_gap=0.02)
xj = sol["xj"]
eval_result = evaluate_allocation("D_B", xj, sol["K_used"], lsoa_master, coords)
elapsed = time.time() - t0

print(f"Scenario B, p=8000 (patched): status={sol['status']}, slack={sol['slack_frac']:.2%}, "
      f"M1={eval_result['M1_avg_dist_m']:.1f}m, M2={eval_result['M2_coverage_800m']:.1%}, time={elapsed:.0f}s")

if sol["status"] != "Optimal":
    print("\nStill not Optimal within 600s -- if this happens, either accept the previous")
    print("11/12-Optimal result with a one-line caveat, or try frac_gap=0.03 for this config only.")


Scenario B, p=8000 (patched): status=Optimal, slack=10.03%, M1=44.9m, M2=99.3%, time=218s


## 3. Overwrite the saved results with the patched solution

In [5]:
results_wide[f"xj_B_p8000"] = pd.Series(xj, index=results_wide.index)
results_wide[f"slack_B_p8000"] = pd.Series(eval_result["sj"], index=results_wide.index)

mask = (run_status["scenario"] == "B") & (run_status["p"] == 8000)
run_status.loc[mask, "status"] = sol["status"]
run_status.loc[mask, "slack_total"] = sol["slack_total"]
run_status.loc[mask, "slack_frac"] = sol["slack_frac"]
run_status.loc[mask, "objective"] = eval_result["objective"]
run_status.loc[mask, "M1_avg_dist_m"] = eval_result["M1_avg_dist_m"]
run_status.loc[mask, "M2_coverage_800m"] = eval_result["M2_coverage_800m"]
run_status.loc[mask, "M3_imd_gap"] = eval_result["M3_imd_gap"]
run_status.loc[mask, "M4_gini"] = eval_result["M4_gini"]

results_wide.to_csv(os.path.join(BASE, "05_processed/p_median_results.csv"), index=False)
run_status.to_csv(os.path.join(BASE, "05_processed/p_median_run_status.csv"), index=False)

print("Saved. Updated status for all 12 configs:")
print(run_status[["scenario", "p", "status", "slack_frac"]].to_string(index=False))
print(f"\nAll Optimal now: {(run_status['status'] == 'Optimal').all()}")


Saved. Updated status for all 12 configs:
scenario    p  status  slack_frac
       A 2000 Optimal    0.378033
       A 5000 Optimal    0.239247
       A 8000 Optimal    0.100461
       B 2000 Optimal    0.377845
       B 5000 Optimal    0.239059
       B 8000 Optimal    0.100273
       C 2000 Optimal    0.377512
       C 5000 Optimal    0.238726
       C 8000 Optimal    0.099940
       D 2000 Optimal    0.377210
       D 5000 Optimal    0.238424
       D 8000 Optimal    0.099638

All Optimal now: True
